In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 84.6 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/ahmedheakl/arazn-whisper-medium

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/ahmedheakl/arazn-whisper-medium)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium",
    device=0,
    chunk_length_s=30,
    generate_kwargs={"language": "ar"}
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [ ]:
pip install word2number


  Preparing metadata (setup.py) ... done
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5568 sha256=80da3e6013a0b1ddd8c82c193cf563ec13cfb6f19333cbbffa5e76d880a81ece
  Stored in directory: /root/.cache/pip/wheels/5b/79/fb/d25928e599c7e11fe4e00d32048cd74933f34a74c633d2aea6
Successfully built word2number
Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install arabic-reshaper

In [ ]:
from word2number import w2n
from arabic_reshaper import reshape

def arabic_to_english_numbers(text: str) -> str:
    """
    Map Arabic number words to English for word2number compatibility.
    """
    arabic_to_english = {
        "صفر": "zero", "واحد": "one", "اثنين": "two", "ثلاثة": "three", "أربعة": "four",
        "خمسة": "five", "ستة": "six", "سبعة": "seven", "ثمانية": "eight", "تسعة": "nine",
        "عشرة": "ten", "مئة": "hundred", "ألف": "thousand", "مليون": "million"
    }
    for ar_num, en_num in arabic_to_english.items():
        text = text.replace(ar_num, en_num)
    return text

In [ ]:
"""
================================================================================
ENHANCED VOICE TO TEXT MODULE FOR CHARITY AID REQUESTS
================================================================================
This version focuses on transcribing and post-processing Egyptian Arabic aid requests, extracting key insights for charity organizations.
"""

import os
import re
import logging
from transformers import pipeline
from typing import Dict, List, Any
from pathlib import Path

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize Hugging Face Pipeline for Transcription
try:
    logger.info("Loading Arazn Whisper Small model...")
    transcribe_pipeline = pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium",
    device=0,
    chunk_length_s=30,
    generate_kwargs={"language": "ar"}
)
    logger.info("Arazn Whisper Small model loaded successfully.")
except Exception as e:
    logger.error(f"Failed to load Arazn Whisper Small model: {str(e)}")
    transcribe_pipeline = None

from pydub.utils import mediainfo

def get_audio_duration(audio_path: str) -> float:
    """
    Get the duration of an audio file in seconds.
    """
    try:
        info = mediainfo(audio_path)
        duration = float(info['duration'])
        logger.info(f"Audio duration: {duration:.2f} seconds.")
        return duration
    except Exception as e:
        logger.error(f"Failed to retrieve audio duration: {e}")
        return 0


from pydub import AudioSegment, effects

def preprocess_audio(audio_path: str, processed_audio_path: str = "processed_audio.wav") -> str:
    """
    Normalize and preprocess the audio file (e.g., denoise, resample).
    """
    audio = AudioSegment.from_file(audio_path)
    normalized_audio = effects.normalize(audio)
    normalized_audio.export(processed_audio_path, format="wav")
    return processed_audio_path



# Define Main Transcription Functions
def transcribe(audio_path: str) -> Dict[str, Any]:
    """
    Transcribe Egyptian Arabic audio and return structured results.
    """
    logger.info(f"Starting transcription for: {audio_path}")

    # Check if pipeline is available
    if not transcribe_pipeline:
        logger.error("Arazn Whisper Small pipeline is not available.")
        return {"error": "Pipeline not available"}

    # Validate audio file exists
    if not os.path.exists(audio_path):
        logger.error(f"Audio file not found: {audio_path}")
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    # Transcribe audio
    try:
        logger.info("Transcribing audio...")
        x=preprocess_audio(audio_path)
        transcription = transcribe_pipeline(x)
        transcribed_text = transcription.get('text', '').strip()
        logger.info("Transcription completed successfully.")
        duration= get_audio_duration(audio_path)

        return {
            "transcribed_text": transcribed_text,
            "language": "Arabic",  # Default to Arabic for the Arazn Whisper model
            "duration_seconds": duration,
            "processing_notes": [f"File: {Path(audio_path).name}"]
        }
    except Exception as e:
        logger.error(f"Error during transcription: {e}")
        return {"error": str(e)}

# -----------------------------------------------------------------------------
# POST-PROCESSING FOR CHARITY AID REQUESTS (ADVANCED)
# -----------------------------------------------------------------------------

# Configure constants for aid context
URGENT_KEYWORDS = ['طارئ', 'عاجل', 'مستعجل', 'دلوقتي', 'ضروري', 'بأسرع وقت', 'محتاج حالاً']
FINANCIAL_WORDS = ['فلوس', 'مال', 'مبلغ', 'دين', 'مساعدة مادية']
MEDICAL_WORDS = ['علاج', 'عملية', 'دواء', 'أدوية', 'مستشفى', 'مرض']
FOOD_WORDS = ['تموين', 'طعام', 'غذاء', 'أكل']
PEOPLE_WORDS = ['أطفال', 'أسرة', 'أم', 'أب', 'عائلة', 'طفل']
THANK_WORDS = ['شكراً', 'ممتن', 'جزاكم الله خير', 'متشكر']

EGYPTIAN_LOCATIONS = ['القاهرة', 'الإسكندرية', 'الجيزة', 'الشرقية', 'سوهاج', 'المنصورة']

# Utility and Extraction Functions
def normalize_egyptian_arabic(text: str) -> str:
    replacements = {
        'عايز': 'أحتاج',
        'محتاج': 'أحتاج',
        'مش': 'ليس',
        'بتاع': 'لـ',
        'دلوقتي': 'الآن',
        'بكره': 'غداً',
        'النهاردة': 'اليوم',
        'حأ': 'سوف',
    }
    for src, tgt in replacements.items():
        text = text.replace(src, tgt)
    return text.strip()

def detect_sentiment(text: str) -> str:
    """
    Detect sentiment for aid requests.
    """
    if any(word in text for word in URGENT_KEYWORDS):
        return "urgent"
    elif any(word in text for word in FINANCIAL_WORDS + MEDICAL_WORDS + FOOD_WORDS):
        return "request"
    elif any(word in text for word in THANK_WORDS):
        return "grateful"
    else:
        return "neutral"

def extract_keywords(text: str, num_keywords: int = 5) -> List[str]:
    """
    Extract important keywords for charity aid requests.
    """
    stop_words = {'في', 'من', 'على', 'عن', 'إلى', 'و', 'أن', 'هذا', 'الذي', 'التي'}
    words = [word.strip(".,!?") for word in text.split() if len(word) > 2 and word not in stop_words]
    unique_words = []
    for word in words:
        if word not in unique_words:
            unique_words.append(word)
    return unique_words[:num_keywords]

def extract_numbers(text: str) -> List[int]:
    """
    Extract numbers from text:
      - Digits
      - Arabic number words converted to integers
    """
    numbers = []

    # 1️⃣ Extract digit numbers
    numbers += [int(n) for n in re.findall(r'\d+', text)]

    # 2️⃣ Extract Arabic number phrases
    words = text.split()
    for i in range(len(words)):
        for j in range(i+1, min(i+6, len(words)+1)):  # check up to 5-word phrases
            phrase = ' '.join(words[i:j])
            try:
                phrase_en = arabic_to_english_numbers(phrase)
                num = w2n.word_to_num(phrase_en)
                numbers.append(num)
            except ValueError:
                continue

    return list(set(numbers))  # remove duplicates


def extract_locations(text: str) -> List[str]:
    """
    Extract mentions of Egyptian locations.
    """
    return [loc for loc in EGYPTIAN_LOCATIONS if loc in text]


def post_process_request(text: str) -> Dict[str, Any]:
    """
    Post-process transcription for charity insights.
    """
    cleaned_text = " ".join(text.split())
    normalized_text = normalize_egyptian_arabic(cleaned_text)
    sentiment = detect_sentiment(normalized_text)
    keywords = extract_keywords(normalized_text)
    numbers = extract_numbers(normalized_text)
    locations = extract_locations(normalized_text)

    return {
        "cleaned_text": cleaned_text,
        "normalized_text": normalized_text,
        "sentiment": sentiment,
        "keywords": keywords,
        "numbers": numbers,
        "locations": locations

    }




INFO:__main__:Loading Arazn Whisper Small model...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-medium/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-medium/abdf7c39ab9d0397620ccaea8974cc764cd0953e/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-medium/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-medium/abdf7c39ab9d0397620ccaea8974cc764cd0953e/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-medium/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-medium/abdf7c39ab9d0397620ccaea8974cc764cd0953e/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai/whisper-medium/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai/whisper-medium/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-medium/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-medium/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/mode

In [ ]:
# Install noisereduce
!pip install noisereduce

# Also install these useful audio processing packages
!pip install librosa
!pip install scipy
!pip install soundfile

In [ ]:
!pip install transformers pydub word2number jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.7 MB/s eta 0:00:0000:0100:01


In [ ]:
import os
import json
import csv
from jiwer import wer, cer

# Folder containing both audios and reference texts
DATASET_FOLDER = "/kaggle/input/datasets/mariemmahmoud5005/voice-text"

# Collect all WAV files
audio_files = [f for f in os.listdir(DATASET_FOLDER) if f.endswith(".wav")]
audio_files.sort()

results = []
total_wer = 0
total_cer = 0
total_number_accuracy = 0

for audio_file in audio_files:
    audio_path = os.path.join(DATASET_FOLDER, audio_file)
    reference_file = audio_file.replace(".wav", ".txt")
    reference_path = os.path.join(DATASET_FOLDER, reference_file)

    if not os.path.exists(reference_path):
        print(f"Reference file not found for {audio_file}, skipping...")
        continue

    # Load reference text
    with open(reference_path, "r", encoding="utf-8") as f:
        reference_text = f.read().strip()

    # Transcribe audio
    transcription_result = transcribe(audio_path)
    predicted_text = transcription_result.get("transcribed_text", "")

    # Post-process
    insights = post_process_request(predicted_text)

    # Compute WER and CER
    sample_wer = wer(reference_text, predicted_text)
    sample_cer = cer(reference_text, predicted_text)
    total_wer += sample_wer
    total_cer += sample_cer

    # Compute number extraction accuracy
    # Extract digits from reference
    ref_numbers = [int(n) for n in re.findall(r'\d+', reference_text)]
    pred_numbers = insights["numbers"]
    correct_numbers = set(ref_numbers) & set(pred_numbers)
    number_accuracy = len(correct_numbers) / max(len(ref_numbers), 1)
    total_number_accuracy += number_accuracy

    # Save per-file results
    results.append({
        "audio_file": audio_file,
        "reference": reference_text,
        "predicted": predicted_text,
        "wer": sample_wer,
        "cer": sample_cer,
        "number_accuracy": number_accuracy,
        "numbers_extracted": pred_numbers
    })

# Compute averages
n_files = len(results)
avg_wer = total_wer / n_files if n_files > 0 else 0
avg_cer = total_cer / n_files if n_files > 0 else 0
avg_number_accuracy = total_number_accuracy / n_files if n_files > 0 else 0

print(f"Average WER: {avg_wer:.3f}")
print(f"Average CER: {avg_cer:.3f}")
print(f"Average Number Accuracy: {avg_number_accuracy:.3f}")

# Save results
json_path = "/kaggle/working/evaluation_results.json"
csv_path = "/kaggle/working/evaluation_results.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

with open(csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    for row in results:
        writer.writerow(row)

print(f"Evaluation saved to {json_path} and {csv_path}")


INFO:__main__:Starting transcription for: /kaggle/input/datasets/mariemmahmoud5005/voice-text/segment.wav
INFO:__main__:Enhancing and transcribing audio...
INFO:__main__:Audio enhanced and saved to: enhanced_audio.wav
INFO:__main__:Transcription completed successfully.
INFO:__main__:Audio duration: 17.00 seconds.
INFO:__main__:Starting transcription for: /kaggle/input/datasets/mariemmahmoud5005/voice-text/segment10.wav
INFO:__main__:Enhancing and transcribing audio...
INFO:__main__:Audio enhanced and saved to: enhanced_audio.wav
INFO:__main__:Transcription completed successfully.
INFO:__main__:Audio duration: 40.00 seconds.
INFO:__main__:Starting transcription for: /kaggle/input/datasets/mariemmahmoud5005/voice-text/segment11.wav
INFO:__main__:Enhancing and transcribing audio...
INFO:__main__:Audio enhanced and saved to: enhanced_audio.wav


KeyboardInterrupt: 

In [ ]:
"""
================================================================================
ENHANCED VOICE TO TEXT MODULE FOR CHARITY AID REQUESTS
================================================================================
This version focuses on transcribing and post-processing Egyptian Arabic aid requests, extracting key insights for charity organizations.
"""

import os
import re
import logging
import numpy as np
from transformers import pipeline
from typing import Dict, List, Any
from pathlib import Path

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize Hugging Face Pipeline for Transcription
try:
    logger.info("Loading Egyptian Arabic Wav2Vec2 model...")
    transcribe_pipeline = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium",
    device=0,
    chunk_length_s=30,
    generate_kwargs={"language": "ar"}
)
    logger.info("Model loaded successfully.")
except Exception as e:
    logger.error(f"Failed to load model: {str(e)}")
    transcribe_pipeline = None

from pydub.utils import mediainfo
from pydub import AudioSegment, effects
from word2number import w2n

def get_audio_duration(audio_path: str) -> float:
    """
    Get the duration of an audio file in seconds.
    """
    try:
        info = mediainfo(audio_path)
        duration = float(info['duration'])
        logger.info(f"Audio duration: {duration:.2f} seconds.")
        return duration
    except Exception as e:
        logger.error(f"Failed to retrieve audio duration: {e}")
        return 0


def simple_audio_enhancement(audio_path: str, output_path: str = "enhanced_audio.wav") -> str:
    """
    Simple audio preprocessing without noisereduce dependency.
    """
    try:
        # Load audio
        audio = AudioSegment.from_file(audio_path)

        # Apply simple preprocessing
        # 1. Normalize volume
        normalized = effects.normalize(audio)

        # 2. Remove very low frequencies (rumble)
        filtered = normalized.high_pass_filter(80)

        # 3. Reduce high frequencies (hiss)
        filtered = filtered.low_pass_filter(8000)

        # 4. Compress dynamic range
        filtered = filtered.compress_dynamic_range()

        # Export enhanced audio
        filtered.export(output_path, format="wav")
        logger.info(f"Audio enhanced and saved to: {output_path}")
        return output_path

    except Exception as e:
        logger.error(f"Audio enhancement failed: {e}")
        # Return original path if enhancement fails
        return audio_path


# Define Arabic number conversion
EGYPTIAN_TO_ENGLISH_NUMS = {
    "واحد": "one", "اثنين": "two", "اتنين": "two",
    "تلاتة": "three", "ثلاثة": "three", "أربعة": "four", "اربعة": "four",
    "خمسة": "five", "ستة": "six", "سبعة": "seven",
    "ثمانية": "eight", "تمانية": "eight", "تسعة": "nine",
    "عشرة": "ten", "عشر": "ten",
    "مئة": "hundred", "مائة": "hundred", "ميت": "hundred",
    "ألف": "thousand", "الف": "thousand",
    "ألفين": "two thousand", "الفين": "two thousand",
    "مليون": "million"
}

def arabic_to_english_numbers(text: str) -> str:
    """
    Convert Arabic number words to English for word2number compatibility.
    """
    # First, replace specific number words
    for ar, en in EGYPTIAN_TO_ENGLISH_NUMS.items():
        text = re.sub(r'\b' + re.escape(ar) + r'\b', en, text)
    return text


# Define Main Transcription Functions
def transcribe(audio_path: str) -> Dict[str, Any]:
    """
    Transcribe Egyptian Arabic audio and return structured results.
    """
    logger.info(f"Starting transcription for: {audio_path}")

    # Check if pipeline is available
    if not transcribe_pipeline:
        logger.error("Transcription pipeline is not available.")
        return {"error": "Pipeline not available"}

    # Validate audio file exists
    if not os.path.exists(audio_path):
        logger.error(f"Audio file not found: {audio_path}")
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    # Transcribe audio
    try:
        logger.info("Enhancing and transcribing audio...")

        # Enhance audio first
        enhanced_path = simple_audio_enhancement(audio_path)

        # Transcribe enhanced audio
        transcription = transcribe_pipeline(enhanced_path)
        transcribed_text = transcription.get('text', '').strip()

        logger.info("Transcription completed successfully.")
        duration = get_audio_duration(audio_path)

        # Clean up temporary enhanced file
        if enhanced_path != audio_path and os.path.exists(enhanced_path):
            try:
                os.remove(enhanced_path)
            except:
                pass

        return {
            "transcribed_text": transcribed_text,
            "language": "Arabic",
            "duration_seconds": duration,
            "processing_notes": [f"File: {Path(audio_path).name}", "Audio enhanced before transcription"]
        }

    except Exception as e:
        logger.error(f"Error during transcription: {e}")
        return {"error": str(e)}


# -----------------------------------------------------------------------------
# POST-PROCESSING FOR CHARITY AID REQUESTS (ADVANCED)
# -----------------------------------------------------------------------------

# Configure constants for aid context
URGENT_KEYWORDS = ['طارئ', 'عاجل', 'مستعجل', 'دلوقتي', 'ضروري', 'بأسرع وقت', 'محتاج حالاً']
FINANCIAL_WORDS = ['فلوس', 'مال', 'مبلغ', 'دين', 'مساعدة مادية']
MEDICAL_WORDS = ['علاج', 'عملية', 'دواء', 'أدوية', 'مستشفى', 'مرض']
FOOD_WORDS = ['تموين', 'طعام', 'غذاء', 'أكل']
PEOPLE_WORDS = ['أطفال', 'أسرة', 'أم', 'أب', 'عائلة', 'طفل']
THANK_WORDS = ['شكراً', 'ممتن', 'جزاكم الله خير', 'متشكر']

EGYPTIAN_LOCATIONS = ['القاهرة', 'الإسكندرية', 'الجيزة', 'الشرقية', 'سوهاج', 'المنصورة']

# Utility and Extraction Functions
def normalize_egyptian_arabic(text: str) -> str:
    replacements = {
        'عايز': 'أحتاج',
        'محتاج': 'أحتاج',
        'مش': 'ليس',
        'بتاع': 'لـ',
        'دلوقتي': 'الآن',
        'بكره': 'غداً',
        'النهاردة': 'اليوم',
        'حأ': 'سوف',
    }
    for src, tgt in replacements.items():
        text = text.replace(src, tgt)
    return text.strip()


def detect_sentiment(text: str) -> str:
    """
    Detect sentiment for aid requests.
    """
    if any(word in text for word in URGENT_KEYWORDS):
        return "urgent"
    elif any(word in text for word in FINANCIAL_WORDS + MEDICAL_WORDS + FOOD_WORDS):
        return "request"
    elif any(word in text for word in THANK_WORDS):
        return "grateful"
    else:
        return "neutral"


def extract_keywords(text: str, num_keywords: int = 5) -> List[str]:
    """
    Extract important keywords for charity aid requests.
    """
    stop_words = {'في', 'من', 'على', 'عن', 'إلى', 'و', 'أن', 'هذا', 'الذي', 'التي'}
    words = [word.strip(".,!?") for word in text.split() if len(word) > 2 and word not in stop_words]
    unique_words = []
    for word in words:
        if word not in unique_words:
            unique_words.append(word)
    return unique_words[:num_keywords]


def extract_numbers(text: str) -> List[int]:
    """
    Extract numbers from text:
      - Digits
      - Arabic number words converted to integers
    """
    numbers = []

    # 1️⃣ Extract digit numbers
    numbers += [int(n) for n in re.findall(r'\d+', text)]

    # 2️⃣ Extract Arabic number words
    # First convert Arabic number words to English
    text_eng = arabic_to_english_numbers(text)

    # Try to extract numbers from the converted text
    try:
        # Simple word extraction
        words = text_eng.split()
        for i, word in enumerate(words):
            # Try to convert single words
            try:
                num = w2n.word_to_num(word)
                numbers.append(num)
            except ValueError:
                # Try multi-word numbers
                for j in range(1, 4):  # Try up to 3-word numbers
                    if i + j <= len(words):
                        phrase = ' '.join(words[i:i+j])
                        try:
                            num = w2n.word_to_num(phrase)
                            numbers.append(num)
                        except ValueError:
                            continue
    except Exception as e:
        logger.error(f"Error extracting numbers: {e}")

    return list(set(numbers))  # remove duplicates


def extract_locations(text: str) -> List[str]:
    """
    Extract mentions of Egyptian locations.
    """
    return [loc for loc in EGYPTIAN_LOCATIONS if loc in text]


def post_process_request(text: str) -> Dict[str, Any]:
    """
    Post-process transcription for charity insights.
    """
    cleaned_text = " ".join(text.split())
    normalized_text = normalize_egyptian_arabic(cleaned_text)
    sentiment = detect_sentiment(normalized_text)
    keywords = extract_keywords(normalized_text)
    numbers = extract_numbers(normalized_text)
    locations = extract_locations(normalized_text)

    return {
        "cleaned_text": cleaned_text,
        "normalized_text": normalized_text,
        "sentiment": sentiment,
        "keywords": keywords,
        "numbers": numbers,
        "locations": locations
    }

def save_result_to_json(result: Dict[str, Any], output_path: str) -> None:
    """
    Save transcription and post-processing result to a JSON file.
    """
    try:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        logger.info(f"Results saved to: {output_path}")
    except Exception as e:
        logger.error(f"Failed to save JSON: {e}")


# -----------------------------------------------------------------------------
# MAIN ENTRY POINT
# -----------------------------------------------------------------------------
if __name__ == '__main__':
    import json

    # Install required packages if not already installed
    try:
        from word2number import w2n
    except ImportError:
        logger.info("Installing word2number...")
        import subprocess
        subprocess.check_call(["pip", "install", "word2number"])
        from word2number import w2n

    try:
        from pydub import AudioSegment
    except ImportError:
        logger.info("Installing pydub...")
        import subprocess
        subprocess.check_call(["pip", "install", "pydub"])
        from pydub import AudioSegment

    audio_file = "test5.wav"  # Replace with the path to an audio file.

    if os.path.exists(audio_file):
        try:
            logger.info("Processing audio file for transcription and insights...")

            # Transcription
            transcription = transcribe(audio_file)

            if "transcribed_text" in transcription:
                logger.info("Post-processing transcription...")
                request_insights = post_process_request(transcription["transcribed_text"])
                full_result = {**transcription, **request_insights}

                audio_name = Path(audio_file).stem
                output_json_path = f"/kaggle/working/{audio_name}_result.json"

                save_result_to_json(full_result, output_json_path)

                print("\nSaved Result:\n")
                print(json.dumps(full_result, indent=2, ensure_ascii=False))
            else:
                logger.error("No transcription text found. Review transcription result.")

        except Exception as e:
            logger.error(f"An error occurred: {e}")
    else:
        logger.error(f"Audio file not found: {audio_file}")

INFO:__main__:Loading Egyptian Arabic Wav2Vec2 model...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-medium/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-medium/abdf7c39ab9d0397620ccaea8974cc764cd0953e/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-medium/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-medium/abdf7c39ab9d0397620ccaea8974cc764cd0953e/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-medium/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-medium/abdf7c39ab9d0397620ccaea8974cc764cd0953e/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai/whisper-medium/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai/whisper-medium/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-medium/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-medium/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/mode

In [ ]:
# Merge both datasets
DATASET_FOLDERS = [
    "/kaggle/input/datasets/mariemmahmoud5005/arabic-charity-speech-dataset",
    "/kaggle/input/datasets/mariemmahmoud5005/voice-text"
]

audio_paths = []

for folder in DATASET_FOLDERS:
    if os.path.exists(folder):
        files = [
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.endswith(".wav")
        ]
        audio_paths.extend(files)

print(f"Total audio files found: {len(audio_paths)}")

Total audio files found: 37


In [ ]:
from jiwer import wer, cer
from difflib import SequenceMatcher
def normalize_for_eval(text: str) -> str:
    """Light Arabic normalization for fair WER/CER"""
    replacements = {
        "أ": "ا", "إ": "ا", "آ": "ا",
        "ى": "ي", "ة": "ه",
        "ؤ": "و", "ئ": "ي"
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text.strip()


def compute_similarity(ref_text: str, hyp_text: str) -> float:
    """Character-based similarity (0–1)"""
    return SequenceMatcher(None, ref_text, hyp_text).ratio()


audio_files = sorted(audio_paths)

results = []
total_wer = total_cer = total_similarity = total_number_accuracy = total_bertscore = 0.0
valid_files = 0

for audio_path in audio_files:
    audio_file = os.path.basename(audio_path)
    ref_path = audio_path.replace(".wav", ".txt")

    if not os.path.exists(ref_path):
        print(f"Missing reference for {audio_file}, skipping.")
        continue

    with open(ref_path, "r", encoding="utf-8") as f:
        reference_text = f.read().strip()

    # Transcribe
    transcription = transcribe(audio_path)
    predicted_text = transcription.get("transcribed_text", "").strip()

    if not predicted_text:
        continue

    # Normalize for evaluation
    ref_norm = normalize_for_eval(reference_text)
    pred_norm = normalize_for_eval(predicted_text)

    # Post-process
    ref_insights = post_process_request(reference_text)
    pred_insights = post_process_request(predicted_text)

    # Metrics
    sample_wer = wer(ref_norm, pred_norm)
    sample_cer = cer(ref_norm, pred_norm)
    similarity = compute_similarity(ref_norm, pred_norm)
    bert_f1 = compute_bertscore_arabic(ref_norm, pred_norm)

    # Number accuracy (FIXED ✅)
    ref_numbers = set(ref_insights["numbers"])
    pred_numbers = set(pred_insights["numbers"])

    if len(ref_numbers) == 0:
        number_accuracy = 1.0  # no numbers to predict
    else:
        number_accuracy = len(ref_numbers & pred_numbers) / len(ref_numbers)

    # Accumulate
    total_wer += sample_wer
    total_cer += sample_cer
    total_similarity += similarity
    total_number_accuracy += number_accuracy
    total_bertscore += bert_f1
    valid_files += 1

    results.append({
        "audio": audio_file,
        "reference": reference_text,
        "prediction": predicted_text,
        "wer": round(sample_wer, 3),
        "cer": round(sample_cer, 3),
        "similarity": round(similarity, 3),
        "number_accuracy": round(number_accuracy, 3),
        "ref_numbers": list(ref_numbers),
        "pred_numbers": list(pred_numbers),
        "arabic_bertscore": round(bert_f1, 3)
    })

# Averages
print("\n===== FINAL METRICS =====")
print(f"Average WER: {total_wer / valid_files:.3f}")
print(f"Average CER: {total_cer / valid_files:.3f}")
print(f"Average Similarity: {total_similarity / valid_files:.3f}")
print(f"Average Number Accuracy: {total_number_accuracy / valid_files:.3f}")
print(f"Average Arabic BERTScore: {total_bertscore / valid_files:.3f}")

# Save outputs
with open("/kaggle/working/evaluation_results_new.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Results saved to /kaggle/working/evaluation_results.json")


INFO:__main__:Starting transcription for: /kaggle/input/datasets/mariemmahmoud5005/arabic-charity-speech-dataset/segment_00-00-05_00-02-01.wav
INFO:__main__:Enhancing and transcribing audio...
INFO:__main__:Audio enhanced and saved to: enhanced_audio.wav
INFO:__main__:Transcription completed successfully.
INFO:__main__:Audio duration: 116.00 seconds.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/bert-base-multilingual-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-multilingual-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
IN

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:__main__:Starting transcription for: /kaggle/input/datasets/mariemmahmoud5005/arabic-charity-speech-dataset/segment_00-00-05_00-02-1.wav
INFO:__main__:Enhancing and transcribing audio...
INFO:__main__:Audio enhanced and saved to: enhanced_audio

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:__main__:Starting transcription for: /kaggle/input/datasets/mariemmahmoud5005/arabic-charity-speech-dataset/segment_00-00-07_00-04-15.wav
INFO:__main__:Enhancing and transcribing audio...
INFO:__main__:Audio enhanced and saved to: enhanced_audi

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:__main__:Starting transcription for: /kaggle/input/datasets/mariemmahmoud5005/voice-text/segment9.wav
INFO:__main__:Enhancing and transcribing audio...
INFO:__main__:Audio enhanced and saved to: enhanced_audio.wav
INFO:__main__:Transcription co

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== FINAL METRICS =====
Average WER: 0.796
Average CER: 0.610
Average Similarity: 0.174
Average Number Accuracy: 0.424
Average Arabic BERTScore: 0.736
Results saved to /kaggle/working/evaluation_results.json


In [ ]:
!pip install bert-score transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.2

In [ ]:
from bert_score import score as bertscore_score

In [ ]:
def compute_bertscore_arabic(ref_text, pred_text):
    """
    Compute Arabic semantic similarity using multilingual BERTScore
    Returns F1 score (0–1)
    """
    try:
        P, R, F1 = bertscore_score(
            [pred_text],
            [ref_text],
            lang="ar",
            model_type="bert-base-multilingual-cased",
            verbose=False
        )
        return F1.item()
    except Exception as e:
        print(f"BERTScore error: {e}")
        return 0.0